# Week 14, ReAct Agent from Scratch (no framework)

```text
# Requirements: pip install numpy pandas openai
```

> ⚠️ REQUIRES: `OPENAI_API_KEY` or `OPENROUTER_API_KEY` (set as environment variables).
> With **no key**, a deterministic **mock brain** drives the same loop, so every cell still runs and the score still prints.

Hand-roll a ReAct agent: **perceive → plan → act → observe → (reflect) → answer**, with guardrails (max steps, cost cap, refusal) and a saved JSONL trace of every run.

## The loop

An agent is a loop where the model's own output decides the next step. This notebook implements the loop **by hand**, no LangGraph, no smolagents, so you own the mental model before any framework hides it.

- **Plan**, the model states a short plan and the first action.
- **Act**, the runtime validates arguments, calls the tool, and returns the result.
- **Observe**, the result is appended to context.
- **Reflect**, after a tool error, the model critiques what went wrong and retries.
- **Answer / Refuse**, the loop ends with a final answer, or a refusal for out-of-scope asks.

Guardrails: a **max-step cap**, a **cost cap** (estimated tokens), and a **refusal path**.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
# Robustness: find the repo root (the directory that contains zoro/) so the import
# works no matter which directory Jupyter was launched from.
_root = pathlib.Path.cwd()
while not (_root / "zoro").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import json, os, re, tempfile
import numpy as np
from zoro import data

SEED = 42
rng = np.random.default_rng(SEED)
print("zoro imported OK · numpy", np.__version__)

## Tools

Four tools, all deterministic and seeded. `track_shipment` reads the synthetic dataset; the others are pure functions. Notice each tool returns a plain `dict` and reports failures as an `"error"` key, the loop and the model both read that key to decide whether to reflect and retry.

In [ ]:
# Build small, seeded lookup tables from the synthetic dataset.
_ship = data.shipments(n=2_000, seed=SEED)
_lanes = data.lanes(seed=11)
_car = data.carriers(seed=7)
_ship_by_id = {row.shipment_id: row for row in _ship.itertuples()}
_lane_by_id = {row.lane_id: row for row in _lanes.itertuples()}
_car_by_id = {row.carrier_id: row for row in _car.itertuples()}
_policies = {d["doc_id"]: d for d in data.policy_docs()}

def track_shipment(shipment_id):
    # Look up a shipment's status, route, delay, and on-time prediction.
    sid = str(shipment_id).strip().upper()
    row = _ship_by_id.get(sid)
    if row is None:
        return {"error": "shipment " + sid + " not found"}
    lane = _lane_by_id.get(row.lane_id)
    carrier = _car_by_id.get(row.carrier_id)
    return {
        "shipment_id": sid,
        "status": row.status,
        "carrier": carrier.carrier_name if carrier else row.carrier_id,
        "origin": lane.origin if lane else "?",
        "destination": lane.destination if lane else "?",
        "planned_arrival": str(row.planned_arrival),
        "delay_hours": round(float(row.delay_hours), 1),
        "on_time": bool(row.is_on_time),
        "weather": row.weather_severity,
    }

def convert_units(value, from_unit, to_unit):
    # Convert between kg/lb, km/mi, and hours/days.
    v = float(value)
    fu = str(from_unit).strip().lower()
    tu = str(to_unit).strip().lower()
    factors = {
        ("kg", "lb"): 2.20462, ("lb", "kg"): 1 / 2.20462,
        ("km", "mi"): 0.621371, ("mi", "km"): 1 / 0.621371,
        ("hours", "days"): 1 / 24, ("days", "hours"): 24,
    }
    if (fu, tu) not in factors:
        return {"error": "unsupported conversion " + from_unit + " -> " + to_unit}
    return {"value": round(v * factors[(fu, tu)], 4), "from_unit": from_unit, "to_unit": to_unit}

_ALLOWED = set("0123456789.+-*/() ")

def calculator(expression):
    # Evaluate a simple arithmetic expression. Only digits and + - * / ( ) . are allowed.
    expr = str(expression).strip()
    if not expr:
        return {"error": "empty expression"}
    if len(expr) > 200:
        return {"error": "expression too long"}
    if any(ch not in _ALLOWED for ch in expr):
        return {"error": "expression contains disallowed characters"}
    try:
        result = eval(expr, {"__builtins__": {}}, {})
        return {"expression": expr, "result": float(result)}
    except Exception as e:
        return {"error": "could not evaluate: " + str(e)}

def get_policy(doc_id):
    # Return a ZoroLogistics policy document by id (POL-001..POL-004).
    key = str(doc_id).strip().upper()
    doc = _policies.get(key)
    if doc is None:
        return {"error": "no policy with id " + key + "; available: " + str(list(_policies))}
    return {"doc_id": doc["doc_id"], "title": doc["title"], "text": doc["text"]}

print("tools defined: track_shipment, convert_units, calculator, get_policy")

In [ ]:
# The tool registry: each entry carries a JSON Schema the model reads as the contract.
TOOLS = {
    "track_shipment": {
        "function": track_shipment,
        "name": "track_shipment",
        "description": "Look up a shipment's status, carrier, route, delay and on-time prediction by shipment id (e.g. S0000123).",
        "parameters": {
            "type": "object",
            "properties": {"shipment_id": {"type": "string", "description": "Shipment id like S0000123"}},
            "required": ["shipment_id"],
        },
    },
    "convert_units": {
        "function": convert_units,
        "name": "convert_units",
        "description": "Convert a value between units: kg<->lb, km<->mi, hours<->days.",
        "parameters": {
            "type": "object",
            "properties": {
                "value": {"type": "number", "description": "Numeric value to convert"},
                "from_unit": {"type": "string", "description": "Source unit (kg, lb, km, mi, hours, days)"},
                "to_unit": {"type": "string", "description": "Target unit"},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
    "calculator": {
        "function": calculator,
        "name": "calculator",
        "description": "Evaluate a simple arithmetic expression using only digits and + - * / ( ) .",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "Arithmetic expression, e.g. 15*3+2"}},
            "required": ["expression"],
        },
    },
    "get_policy": {
        "function": get_policy,
        "name": "get_policy",
        "description": "Return a ZoroLogistics policy document by id (POL-001 shipping, POL-002 refunds, POL-003 dangerous goods, POL-004 customs).",
        "parameters": {
            "type": "object",
            "properties": {"doc_id": {"type": "string", "description": "Policy id like POL-002"}},
            "required": ["doc_id"],
        },
    },
}

print("tool schema example (calculator):")
print(json.dumps({k: v for k, v in TOOLS["calculator"].items() if k != "function"}, indent=2))

In [ ]:
# Smoke-test each tool deterministically.
print(track_shipment("S0000123"))
print(convert_units(500, "kg", "lb"))
print(calculator("15*3+2"))
print(get_policy("POL-002")["title"])
print(track_shipment("DOES_NOT_EXIST"))  # an error result, for the reflection demo later

## Cost tracker & token estimator

Cost caps are a guardrail, not an afterthought. We estimate tokens as `len(text) // 4` (a rough 4-char/token heuristic) and price input/output separately, so the loop can refuse to spend past a budget.

In [ ]:
class CostTracker:
    # Simple token estimator + dollar accounting. Prices are illustrative (per 1M tokens).
    def __init__(self, input_per_m=2.50, output_per_m=10.0, cap_usd=0.05):
        self.input_per_m = input_per_m
        self.output_per_m = output_per_m
        self.cap = cap_usd
        self.input_tokens = 0
        self.output_tokens = 0

    @staticmethod
    def estimate(text):
        return max(1, len(str(text)) // 4)

    def count_input(self, text):
        self.input_tokens += self.estimate(text)

    def count_output(self, text):
        self.output_tokens += self.estimate(text)

    def cost(self):
        return self.input_tokens / 1e6 * self.input_per_m + self.output_tokens / 1e6 * self.output_per_m

    def over(self):
        return self.cost() >= self.cap

    def total_tokens(self):
        return self.input_tokens + self.output_tokens

print("cost tracker ready (cap $0.05 by default)")

## The LLM client + a deterministic mock brain

The same loop runs with either brain:

- **Real brain**, calls OpenAI (or OpenRouter with `OPENROUTER_API_KEY`) and parses a JSON object.
- **Mock brain**, a deterministic keyword classifier that emits the same JSON steps, so the loop, tools, guardrails, and scoring all run with no key and no cost.

In [ ]:
_openai = None
try:
    import openai as _openai
except Exception:
    _openai = None

def make_client():
    # Returns (client, model) or (None, None) when no key is present.
    if _openai is None:
        return None, None
    key = os.environ.get("OPENAI_API_KEY")
    base = None
    model = "gpt-4o-mini"
    if not key:
        key = os.environ.get("OPENROUTER_API_KEY")
        if key:
            base = "https://openrouter.ai/api/v1"
            model = "openai/gpt-4o-mini"
    if not key:
        return None, None
    client = _openai.OpenAI(api_key=key, base_url=base) if base else _openai.OpenAI(api_key=key)
    return client, model

def system_prompt():
    tool_desc = "\n".join("- " + t["name"] + ": " + t["description"] for t in TOOLS.values())
    return (
        "You are Zoro, a support agent for ZoroLogistics (a freight company).\n"
        "Respond with a SINGLE JSON object only, no prose, choosing exactly one kind:\n"
        '{"kind":"plan","plan":"<one sentence>","action":{"tool":"<name>","args":{...}}}\n'
        '{"kind":"action","tool":"<name>","args":{...}}\n'
        '{"kind":"reflect","reflection":"<what went wrong and what to change>"}\n'
        '{"kind":"answer","answer":"<final answer>"}\n'
        '{"kind":"refuse","reason":"<why you will not or cannot help>"}\n'
        "Available tools:\n" + tool_desc + "\n"
        "Rules: use tools for anything requiring live ZoroLogistics data; never invent shipment facts. "
        "If a request is outside ZoroLogistics (deleting data, general knowledge, weather), refuse with kind refuse. "
        "If a tool errors, reflect then retry with corrected arguments."
    )

def parse_json_action(text):
    if not text:
        return {"kind": "reflect", "reflection": "empty model output"}
    s = str(text).strip().strip("`")
    if s.startswith("json"):
        s = s[4:].strip()
    start, end = s.find("{"), s.rfind("}")
    if start == -1 or end <= start:
        return {"kind": "reflect", "reflection": "model did not return JSON: " + s[:120]}
    try:
        return json.loads(s[start:end + 1])
    except Exception:
        return {"kind": "reflect", "reflection": "model returned invalid JSON: " + s[start:end + 1][:120]}

def real_brain(client, model):
    def brain(messages):
        resp = client.chat.completions.create(model=model, messages=messages, temperature=0.0)
        return parse_json_action(resp.choices[0].message.content)
    return brain

def _extract_expr(query):
    expr = "".join(ch for ch in query if ch in "0123456789.+-*/()")
    return expr or "0"

def _classify_mock(query):
    low = query.lower()
    if "delete all" in low or "weather" in low:
        return {"refuse": True}
    if "lithium" in low or "battery" in low:
        return {"tool": "get_policy", "args": {"doc_id": "POL-003"}}
    if "refund" in low and "policy" in low:
        return {"tool": "get_policy", "args": {"doc_id": "POL-002"}}
    if "freight cost" in low:
        return {"tool": "calculator", "args": {"expression": "1.35*10*800"}}
    if "convert" in low and "pounds" in low:
        m = re.search(r"\d+(?:\.\d+)?", query)
        val = float(m.group(0)) if m else 500.0
        if val.is_integer():
            val = int(val)
        return {"tool": "convert_units", "args": {"value": val, "from_unit": "kg", "to_unit": "lb"}}
    if "miles" in low and "kilometer" in low:
        m = re.search(r"\d+(?:\.\d+)?", query)
        val = float(m.group(0)) if m else 800.0
        if val.is_integer():
            val = int(val)
        return {"tool": "convert_units", "args": {"value": val, "from_unit": "km", "to_unit": "mi"}}
    if re.search(r"S\d{7}", query, re.I):
        sid = re.search(r"S\d{7}", query, re.I).group(0).upper()
        return {"tool": "track_shipment", "args": {"shipment_id": sid}}
    if re.search(r"\d+\s*[+\-*/]\s*\d+", query) or "*" in query:
        return {"tool": "calculator", "args": {"expression": _extract_expr(query)}}
    return {"tool": "track_shipment", "args": {"shipment_id": "S0000001"}}

def make_mock_brain(query):
    intent = _classify_mock(query)
    state = {"n": 0, "reflected": False}
    def brain(messages):
        state["n"] += 1
        last = messages[-1]
        last_is_obs = last.get("role") == "tool"
        if intent.get("refuse"):
            if state["n"] == 1:
                return {"kind": "plan", "plan": "Request is outside ZoroLogistics scope.", "action": None}
            return {"kind": "refuse", "reason": "Out-of-scope request: no tool can satisfy it and acting would violate policy."}
        tool, args = intent["tool"], intent["args"]
        if state["n"] == 1:
            return {"kind": "plan", "plan": "I will answer by calling " + tool + ".", "action": {"tool": tool, "args": args}}
        if last_is_obs:
            content = json.loads(last["content"]) if isinstance(last["content"], str) else last["content"]
            if isinstance(content, dict) and "error" in content and not state["reflected"]:
                state["reflected"] = True
                return {"kind": "reflect", "reflection": tool + " returned an error: " + str(content["error"]) + ". I will retry with corrected arguments."}
            return {"kind": "answer", "answer": "Resolved for: " + query}
        if state["reflected"]:
            return {"kind": "action", "tool": tool, "args": args}
        return {"kind": "answer", "answer": "Done."}
    return brain

client, model = make_client()
USE_REAL = client is not None
print(("Using real model: " + model) if USE_REAL else "No API key found, using the deterministic mock brain (set OPENAI_API_KEY or OPENROUTER_API_KEY for the real model).")

## The ReAct loop

One loop, used by both brains. Every turn is appended to a trace; a tool error becomes an observation the brain can reflect on; the loop stops on `answer`, `refuse`, `max_steps`, or the `cost cap`.

In [ ]:
TRACE_PATH = pathlib.Path(tempfile.gettempdir()) / "zorologistics_w14_traces.jsonl"
if TRACE_PATH.exists():
    TRACE_PATH.unlink()

def execute_tool(action, tracker, trace):
    name = action.get("tool")
    args = action.get("args", {})
    entry = TOOLS.get(name)
    if entry is None:
        result = {"error": "unknown tool " + str(name) + "; available: " + str(list(TOOLS))}
    else:
        try:
            result = entry["function"](**args)
        except TypeError as e:
            result = {"error": "bad arguments for " + str(name) + ": " + str(e)}
        except Exception as e:
            result = {"error": str(name) + " failed: " + str(e)}
    tracker.count_output(json.dumps(result))
    trace.append({"tool": name, "args": args, "result": result})
    return result

def save_trace(run_id, query, trace, final):
    with open(TRACE_PATH, "a") as f:
        f.write(json.dumps({"run_id": run_id, "query": query, "trace": trace, "final": final}) + "\n")

def react_loop(query, brain, max_steps=8, cost_cap=0.05, tracker=None):
    if tracker is None:
        tracker = CostTracker(cap_usd=cost_cap)
    tracker.count_input(system_prompt() + query)
    messages = [
        {"role": "system", "content": system_prompt()},
        {"role": "user", "content": query},
    ]
    trace = []
    final, refused = None, False
    for step in range(max_steps):
        if tracker.over():
            trace.append({"step": "guardrail", "event": "cost_cap", "cost": round(tracker.cost(), 6)})
            final = "I have reached my cost budget and must stop. Please rephrase or try later."
            break
        out = brain(messages)
        kind = out.get("kind")
        trace.append({"step": step, "kind": kind, "out": out})
        tracker.count_output(json.dumps(out))
        if kind == "refuse":
            refused = True
            final = out.get("reason", "Refused.")
            break
        if kind == "answer":
            final = out.get("answer", "")
            break
        if kind == "reflect":
            messages.append({"role": "assistant", "content": "Reflection: " + str(out.get("reflection", ""))})
            continue
        action = out.get("action")
        if kind == "action" and action is None:
            action = {"tool": out.get("tool"), "args": out.get("args", {})}
        if action:
            result = execute_tool(action, tracker, trace)
            messages.append({"role": "assistant", "content": json.dumps(out)})
            messages.append({"role": "tool", "name": action.get("tool"), "content": json.dumps(result)})
        else:
            messages.append({"role": "assistant", "content": json.dumps(out)})
    else:
        trace.append({"step": "guardrail", "event": "max_steps"})
        final = "I reached my step limit before finishing."
    return {"final": final, "refused": refused, "trace": trace, "cost": round(tracker.cost(), 6), "tokens": tracker.total_tokens()}

print("react_loop defined; trace file:", TRACE_PATH)

## Run one scenario and read the trace

This is the point of the week: the trace, not the final answer, is the debugger.

In [ ]:
res = react_loop("Where is shipment S0000123 right now?", make_mock_brain("Where is shipment S0000123 right now?"))
save_trace("demo-1", "Where is shipment S0000123 right now?", res["trace"], res["final"])
print("FINAL:", res["final"])
print("COST: $%.6f · TOKENS: %d" % (res["cost"], res["tokens"]))
print("--- trace ---")
for ev in res["trace"]:
    if "tool" in ev:
        print("  TOOL", ev["tool"], ev["args"], "->", ev["result"])
    elif "kind" in ev:
        print("  STEP", ev["kind"], str(ev["out"])[:80])

## Reflection demo

Force a tool error (`1/0`), watch the brain **reflect**, then **retry** with a valid expression. Reflection is how the agent recovers without human help.

In [ ]:
ref_state = {"phase": 0}

def reflection_brain(messages):
    last = messages[-1]
    if ref_state["phase"] == 0:
        ref_state["phase"] = 1
        return {"kind": "action", "tool": "calculator", "args": {"expression": "1/0"}}
    if ref_state["phase"] == 1 and last.get("role") == "tool":
        ref_state["phase"] = 2
        return {"kind": "reflect", "reflection": "The tool errored (division by zero). I will retry with a valid expression."}
    if ref_state["phase"] == 2:
        ref_state["phase"] = 3
        return {"kind": "action", "tool": "calculator", "args": {"expression": "2 + 2"}}
    if ref_state["phase"] == 3 and last.get("role") == "tool":
        ref_state["phase"] = 4
        return {"kind": "answer", "answer": "2 + 2 = 4"}
    return {"kind": "answer", "answer": "4"}

r2 = react_loop("compute 2 + 2 after a hiccup", reflection_brain, max_steps=6)
print("FINAL:", r2["final"])
for ev in r2["trace"]:
    if "tool" in ev:
        print("  TOOL", ev["tool"], ev["args"], "->", ev["result"])
    elif ev.get("kind"):
        print("  STEP", ev["kind"], str(ev["out"])[:70])

## Guardrails

Three tests, each asserting a guardrail actually fires: **refusal** (out-of-scope), **max steps** (a brain that never terminates), and **cost cap** (a zero budget).

In [ ]:
# 1) refusal
r_refuse = react_loop("Delete all shipment records from the database.", make_mock_brain("Delete all shipment records from the database."))
assert r_refuse["refused"], "refusal did not fire"
print("refusal OK ->", r_refuse["final"])

# 2) max steps: a brain that reflects forever
def endless(messages):
    return {"kind": "reflect", "reflection": "still thinking"}
r_max = react_loop("anything", endless, max_steps=4)
assert any(ev.get("event") == "max_steps" for ev in r_max["trace"]), "max_steps did not fire"
print("max_steps OK (4-step cap) ->", r_max["final"])

# 3) cost cap: a zero budget stops the loop immediately
r_cost = react_loop("Where is S0000123?", make_mock_brain("Where is S0000123?"), cost_cap=0.0)
assert any(ev.get("event") == "cost_cap" for ev in r_cost["trace"]), "cost_cap did not fire"
print("cost_cap OK ->", r_cost["final"])

print("ALL GUARDRAILS FIRED")

## Score over 10 scenarios

Ten unseen scenarios: six tool-use cases, two policy lookups, and two refusals. Correct means *the right tool was called with the right arguments, no error, and a final answer was produced*, or, for the refusal cases, *the agent refused*.

In [ ]:
SCENARIOS = [
    {"id": 1, "query": "Where is shipment S0000123 right now?", "expect": {"tool": "track_shipment", "args": {"shipment_id": "S0000123"}}},
    {"id": 2, "query": "Is shipment S0000456 going to arrive on time?", "expect": {"tool": "track_shipment", "args": {"shipment_id": "S0000456"}}},
    {"id": 3, "query": "Convert 500 kg to pounds.", "expect": {"tool": "convert_units", "args": {"value": 500, "from_unit": "kg", "to_unit": "lb"}}},
    {"id": 4, "query": "How many miles is 800 kilometers?", "expect": {"tool": "convert_units", "args": {"value": 800, "from_unit": "km", "to_unit": "mi"}}},
    {"id": 5, "query": "What is 15 * 3 + 2?", "expect": {"tool": "calculator", "args": {"expression": "15*3+2"}}},
    {"id": 6, "query": "A rate of 1.35 USD per km per ton, 10 tons, 800 km. What is the freight cost?", "expect": {"tool": "calculator", "args": {"expression": "1.35*10*800"}}},
    {"id": 7, "query": "What is the refund policy for a shipment that arrived 60 hours late?", "expect": {"tool": "get_policy", "args": {"doc_id": "POL-002"}}},
    {"id": 8, "query": "Are lithium batteries over 100 Wh allowed to ship?", "expect": {"tool": "get_policy", "args": {"doc_id": "POL-003"}}},
    {"id": 9, "query": "Delete all shipment records from the database.", "expect": {"refuse": True}},
    {"id": 10, "query": "What's the weather forecast in Paris this weekend?", "expect": {"refuse": True}},
]

def score_run(res, expected):
    if expected.get("refuse"):
        return 1 if res["refused"] else 0
    tool, args = expected["tool"], expected["args"]
    for ev in res["trace"]:
        if ev.get("tool") == tool and "result" in ev:
            ok = all(str(ev.get("args", {}).get(k)) == str(v) for k, v in args.items())
            if ok and "error" not in ev["result"] and bool(res.get("final")):
                return 1
    return 0

def run_agent(query):
    brain = real_brain(client, model) if USE_REAL else make_mock_brain(query)
    return react_loop(query, brain)

results, scores = [], []
for i, sc in enumerate(SCENARIOS):
    res = run_agent(sc["query"])
    results.append(res)
    s = score_run(res, sc["expect"])
    scores.append(s)
    save_trace("scenario-%02d" % sc["id"], sc["query"], res["trace"], res["final"])
    print("scenario %2d  %-4s  %s" % (sc["id"], "PASS" if s else "FAIL", sc["query"][:60]))

print()
print("Per-scenario: ", scores)

In [ ]:
final_score = round(sum(scores) / len(SCENARIOS), 4)
print("Correct:", sum(scores), "/", len(SCENARIOS))
print("FINAL_SCORE", final_score)